# Manual Run - ETL iSeller Pusat DWH

> Notebook untuk menjalankan pipeline ETL iSeller Pusat secara manual.
>
> **Flow:**
> 1. Backfill Fulfillment (detect unfulfilled orders → re-pull API → update DL)
> 2. Re-dump 8 DWH Summary Tables secara berurutan
> 3. Kirim email report
>
> Created by data-team

---
## CONFIGURATION

In [ ]:
import datetime
import sys
import os

# ===================== SET MODE =============================
#    True  = Otomatis (kemarin / current_date - 1)
#    False = Manual (isi START_DATE & END_DATE sendiri)
# ============================================================

AUTO_MODE = False

# ============================================================
#  MODE MANUAL: Isi tanggal di bawah (isi hanya jika AUTO_MODE = False)
# ============================================================

MANUAL_START_DATE = '2026-03-28'
MANUAL_END_DATE   = '2026-03-28'

# ===================== SKIP DL CHECK ========================
#    True  = Skip backfill fulfillment (cek unfulfilled order)
#            (gunakan jika DL sudah di-clean manual,
#             langsung re-dump DWH saja + kirim report email)
#    False = Full flow (backfill fulfillment → re-dump DWH)
# ============================================================

SKIP_DL_CHECK = True

# ============================================================

if AUTO_MODE:
    yesterday = (datetime.date.today() - datetime.timedelta(days=1)).strftime("%Y-%m-%d")
    START_DATE = (datetime.date.today() - datetime.timedelta(days=9)).strftime("%Y-%m-%d")
    END_DATE   = yesterday
else:
    START_DATE = MANUAL_START_DATE
    END_DATE   = MANUAL_END_DATE

print(f"Mode      : {'AUTO (yesterday - 8 days window)' if AUTO_MODE else 'MANUAL'}")
print(f"Period    : {START_DATE} s/d {END_DATE}")
print(f"Skip DL   : {SKIP_DL_CHECK}")

---
## LOAD CONFIG & IMPORT PIPELINE

In [ ]:
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dw = module_from_file("config_dw", "../../config/config_db_datawarehouse.py")

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'dags', 'iseller_pusat_dwh', 'pipelines'))
sys.path.insert(0, os.path.join(os.path.abspath('.'), 'pipelines'))

from etl_orchestrator import run_all

print("Config & pipeline loaded successfully.")

---
## RUN PIPELINE

In [ ]:
results = run_all(
    start_date=START_DATE,
    end_date=END_DATE,
    config_dw=config_dw,
    skip_dl_check=SKIP_DL_CHECK
)

---
## REVIEW RESULTS

In [ ]:
if results:
    print(f"{'Table':<55} {'Status':<10} {'Deleted':>10} {'Dumped':>10}")
    print("-" * 90)
    for table_name, r in results.items():
        if r['status'] == 'SUCCESS':
            print(f"{table_name:<55} {r['status']:<10} {r['deleted']:>10,} {r['dumped']:>10,}")
        else:
            print(f"{table_name:<55} {r['status']:<10} Error: {r['error']}")
else:
    print("No results returned.")